In [1]:
# ==========================================================
# Imports
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

from xgboost import XGBClassifier

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [2]:
# ==========================================================
# Load Feature Dataset
# ==========================================================


DATA = "/kaggle/input/notebooks/shri7ul/03-feature-engineering-ipynb/master_feature_engineered.parquet"

master = pd.read_parquet(DATA)

print(master.shape)

master.head()

(35072, 122)


,response_id,session_id,learning_objective_id,learning_objective,transcript,student_text,tutor_text,background_text,is_correct,objective_frequency,log_objective_frequency,is_rare_objective,objective_frequency_percentile,objective_char_count,objective_word_count,objective_avg_word_length,obj_has_fraction,obj_has_decimal,obj_has_money,obj_has_graph,obj_has_table,obj_has_probability,obj_has_ratio,obj_has_angle,obj_has_shape,obj_has_division,obj_has_multiply,obj_has_multiplication,obj_has_addition,obj_has_subtract,obj_has_subtraction,obj_has_equation,obj_has_percentage,obj_has_volume,obj_has_area,obj_has_length,objective_family,transcript_chars,student_chars,tutor_chars,background_chars,transcript_words,student_words,tutor_words,background_words,log_transcript_chars,log_student_chars,log_tutor_chars,log_background_chars,student_fraction,tutor_fraction,background_fraction,tutor_student_ratio,log_tutor_student_ratio,student_minus_tutor,tutor_minus_student,student_turns,tutor_turns,background_turns,total_turns,student_turn_fraction,tutor_turn_fraction,background_turn_fraction,turn_ratio,log_turn_ratio,chars_per_turn,words_per_turn,question_count,student_question_count,tutor_question_count,question_ratio,unclear_count,unclear_ratio,digit_count,math_symbol_count,digit_ratio,uppercase_count,uppercase_ratio,enc_good,enc_great,enc_excellent,enc_awesome,enc_perfect,enc_correct,enc_nice,enc_well_done,enc_brilliant,encouragement_score,inst_try,inst_remember,inst_because,inst_think,inst_first,inst_next,inst_explain,inst_look,inst_tell,instruction_score,stu_dont_know,stu_maybe,stu_i_think,stu_not_sure,stu_guess,student_uncertainty_score,student_words_per_turn,tutor_words_per_turn,questions_per_turn,unclear_per_turn,digits_per_turn,encouragement_per_turn,instruction_per_turn,log_question_count,log_unclear_count,log_digit_count,log_math_symbol_count,log_encouragement_score,log_instruction_score,log_student_uncertainty_score,background_heavy,many_questions,many_unclear,long_session
0,aaaavsh,bcaufvc,dqibnvd,Knowing the value of each digit in numbers wit...,"[BACKGROUND] [unclear]\n[TUTOR] Miss, I can't ...","Can you hear me? Hello? Yeah, I hear you. Can ...","Miss, I can't hear. Can you hear me? [unclear]...","[unclear] Good, very good. [unclear] And we wi...",1.0,1267,7.145196,0,0.942803,71,14,5.071429,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Fractions_Decimals,20829,4877,12188,767,3610,890,2251,139,9.944150,8.492491,9.408289,6.643790,0.234133,0.585118,0.036822,2.498770,1.252411,-7311,7311,150,169,11,330,0.453172,0.510574,0.033233,1.125828,0.754161,62.927492,10.906344,129,24,102,0.035724,98,0.027139,374,12,0.017955,2649,0.127172,6,4,1,2,0,10,1,6,0,30,19,0,2,4,8,7,1,6,7,54,0,0,1,0,0,1,5.894040,13.241176,0.389728,0.296073,1.129909,0.176471,0.317647,4.867534,4.595120,5.926926,2.564949,3.433987,4.007333,0.693147,0,0,1,0
1,aaabhzi,eyutanf,eukmzxl,Adding and subtracting tens to a 2-digit number.,[TUTOR] Yay! Hello!\n[STUDENT] Hello.\n[TUTOR]...,Hello. Hello. I was one minute early. I'm one ...,Yay! Hello! Hello. Hello. [Speaker:Background]...,20. What 2-digit number can you make using the...,1.0,44,3.806662,0,0.061089,49,8,6.125000,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,Arithmetic,29463,9725,16431,754,5495,1918,3146,151,10.290924,9.182558,9.706986,6.626718,0.330064,0.557664,0.025591,1.689492,0.989352,-6706,6706,123,144,13,280,0.437722,0.512456,0.046263,1.169355,0.774430,104.850534,19.555160,122,20,98,0.022198,79,0.014374,491,45,0.016664,2609,0.088549,22,2,0,1,0,7,0,13,3,48,8,3,19,37,4,8,9,4,9,101,0,8,2,0,1,11,15.467742,21.696552,0.434164,0.281139,1.747331,0.331034,0.696552,4.812184,4.382027,6.198479,3.828641,3.891820,4.624973,2.484907,0,0,0,1
2,aaahpnz,juptkxd,fjbqcsv,Comparing and ordering fractions by finding a ...,[BACKGROUND] [unclear]\n[TUTOR] Is it me you a...,Hello. Hello. Yes. Good. How are you? Normal. ...,"Is it me you are looking for? Okay, so did you...","[unclear] Hello. Hello, Callum. Can you hear m...",0.0,211,5.356586,0,0.259466,65,9,7

In [3]:
# ==========================================================
# Prepare Features
# ==========================================================

TARGET = "is_correct"

DROP_COLUMNS = [

    "response_id",
    "session_id",

    "learning_objective",
    "learning_objective_id",

    "transcript",
    "student_text",
    "tutor_text",
    "background_text",

    TARGET,
]

X = master.drop(columns=DROP_COLUMNS)

y = master[TARGET].astype(int)

print(X.shape)

(35072, 113)


In [4]:
# ==========================================================
# Encode Categoricals
# ==========================================================

cat_cols = X.select_dtypes(include="object").columns

for col in cat_cols:
    X[col] = X[col].astype("category").cat.codes

print(cat_cols.tolist())

['objective_family']


In [5]:
learning_rates = [
    0.20,
    0.10,
    0.05,
    0.03,
    0.02,
]

In [6]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

learning_rates = [0.20, 0.10, 0.05, 0.03, 0.02]

results = []

for lr in learning_rates:

    print("="*70)
    print(f"Learning Rate : {lr}")
    print("="*70)

    oof = np.zeros(len(X))
    fold_losses = []

    kf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    for train_idx, valid_idx in kf.split(X, y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = XGBClassifier(

            n_estimators=3000,
            learning_rate=lr,
            max_depth=6,

            subsample=0.8,
            colsample_bytree=0.8,

            objective="binary:logistic",
            eval_metric="logloss",

            random_state=42,

            tree_method="hist",

            early_stopping_rounds=300,
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False
        )

        pred = model.predict_proba(X_valid)[:,1]

        oof[valid_idx] = pred

        fold_losses.append(
            log_loss(y_valid, pred)
        )

    overall = log_loss(y, oof)

    print("Fold Mean :", np.mean(fold_losses))
    print("OOF       :", overall)

    results.append([lr, overall])

Learning Rate : 0.2
Fold Mean : 0.5530750144357736
OOF       : 0.5530749428880292
Learning Rate : 0.1
Fold Mean : 0.547253863799701
OOF       : 0.5472537348721999
Learning Rate : 0.05
Fold Mean : 0.5460832347139869
OOF       : 0.546083168620845
Learning Rate : 0.03
Fold Mean : 0.5454361698047049
OOF       : 0.5454360812667409
Learning Rate : 0.02
Fold Mean : 0.5445509217985086
OOF       : 0.5445508704538526


In [7]:
result_df = pd.DataFrame(
    results,
    columns=[
        "learning_rate",
        "logloss"
    ]
)

result_df = result_df.sort_values("logloss")

display(result_df)

,learning_rate,logloss
4,0.02,0.544551
3,0.03,0.545436
2,0.05,0.546083
1,0.10,0.547254
0,0.20,0.553075


In [8]:
depths = [3, 4, 5, 6, 7, 8]

results = []

for depth in depths:

    print("="*70)
    print(f"Max Depth : {depth}")
    print("="*70)

    oof = np.zeros(len(X))
    losses = []

    kf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    for train_idx, valid_idx in kf.split(X, y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = XGBClassifier(

            n_estimators=3000,

            learning_rate=0.02,

            max_depth=depth,

            subsample=0.8,
            colsample_bytree=0.8,

            objective="binary:logistic",
            eval_metric="logloss",

            random_state=42,

            tree_method="hist",

            early_stopping_rounds=300,
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False,
        )

        pred = model.predict_proba(X_valid)[:,1]

        oof[valid_idx] = pred

        losses.append(log_loss(y_valid, pred))

    overall = log_loss(y, oof)

    print("Fold Mean :", np.mean(losses))
    print("OOF       :", overall)

    results.append([depth, overall])

result_df = pd.DataFrame(
    results,
    columns=[
        "max_depth",
        "logloss"
    ]
).sort_values("logloss")

display(result_df)

Max Depth : 3
Fold Mean : 0.5477091518276944
OOF       : 0.5477090890616286
Max Depth : 4
Fold Mean : 0.5460839620880581
OOF       : 0.5460838887272906
Max Depth : 5
Fold Mean : 0.5453857322572895
OOF       : 0.5453856416187436
Max Depth : 6
Fold Mean : 0.5445509217985086
OOF       : 0.5445508704538526
Max Depth : 7
Fold Mean : 0.5438223445852304
OOF       : 0.5438223061426841
Max Depth : 8
Fold Mean : 0.5444254185279197
OOF       : 0.5444253524350988


,max_depth,logloss
4,7,0.543822
5,8,0.544425
3,6,0.544551
2,5,0.545386
1,4,0.546084
0,3,0.547709


## learning_rate = 0.02  
## max_depth = 7

fix

In [9]:
# ==========================================================
# Tune min_child_weight
# ==========================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

results = []

candidate_values = [
    1,
    2,
    3,
    5,
    7,
    10,
]

for value in candidate_values:

    print("="*60)
    print(f"min_child_weight = {value}")
    print("="*60)

    kf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    oof_pred = np.zeros(len(X))

    for train_idx, valid_idx in kf.split(X, y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = XGBClassifier(

            objective="binary:logistic",
            eval_metric="logloss",

            learning_rate=0.02,
            max_depth=7,
            min_child_weight=value,

            n_estimators=3000,

            subsample=0.8,
            colsample_bytree=0.8,

            random_state=42,
            tree_method="hist",

            early_stopping_rounds=300,
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False,
        )

        pred = model.predict_proba(X_valid)[:,1]

        oof_pred[valid_idx] = pred

    score = log_loss(y, oof_pred)

    results.append({
        "min_child_weight": value,
        "logloss": score,
    })

    print(f"LogLoss : {score:.6f}")
    print()

results = (
    pd.DataFrame(results)
    .sort_values("logloss")
    .reset_index(drop=True)
)

display(results)

min_child_weight = 1
LogLoss : 0.543822

min_child_weight = 2
LogLoss : 0.544180

min_child_weight = 3
LogLoss : 0.544235

min_child_weight = 5
LogLoss : 0.544333

min_child_weight = 7
LogLoss : 0.544112

min_child_weight = 10
LogLoss : 0.544217



,min_child_weight,logloss
0,1,0.543822
1,7,0.544112
2,2,0.544180
3,10,0.544217
4,3,0.544235
5,5,0.544333


**learning_rate = 0.02  
max_depth = 7  
min_child_weight = 1**

In [10]:
# ==========================================================
# Tune subsample
# ==========================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

results = []

candidate_values = [
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
    1.0,
]

for value in candidate_values:

    print("=" * 60)
    print(f"subsample = {value}")
    print("=" * 60)

    kf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    oof_pred = np.zeros(len(X))

    for train_idx, valid_idx in kf.split(X, y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = XGBClassifier(

            objective="binary:logistic",
            eval_metric="logloss",

            learning_rate=0.02,
            max_depth=7,
            min_child_weight=1,

            subsample=value,
            colsample_bytree=0.8,

            n_estimators=3000,

            tree_method="hist",

            random_state=42,

            early_stopping_rounds=300,
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False,
        )

        pred = model.predict_proba(X_valid)[:, 1]

        oof_pred[valid_idx] = pred

    score = log_loss(y, oof_pred)

    results.append({
        "subsample": value,
        "logloss": score,
    })

    print(f"OOF LogLoss : {score:.6f}")
    print()

results = (
    pd.DataFrame(results)
    .sort_values("logloss")
    .reset_index(drop=True)
)

display(results)

subsample = 0.5
OOF LogLoss : 0.545796

subsample = 0.6
OOF LogLoss : 0.545665

subsample = 0.7
OOF LogLoss : 0.544673

subsample = 0.8
OOF LogLoss : 0.543822

subsample = 0.9
OOF LogLoss : 0.544103

subsample = 1.0
OOF LogLoss : 0.545981



,subsample,logloss
0,0.8,0.543822
1,0.9,0.544103
2,0.7,0.544673
3,0.6,0.545665
4,0.5,0.545796
5,1.0,0.545981


In [11]:
# ==========================================================
# Tune colsample_bytree
# ==========================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from xgboost import XGBClassifier

results = []

candidate_values = [
    0.5,
    0.6,
    0.7,
    0.8,
    0.9,
    1.0,
]

for value in candidate_values:

    print("=" * 60)
    print(f"colsample_bytree = {value}")
    print("=" * 60)

    kf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    oof_pred = np.zeros(len(X))

    for train_idx, valid_idx in kf.split(X, y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = XGBClassifier(

            objective="binary:logistic",
            eval_metric="logloss",

            learning_rate=0.02,
            max_depth=7,
            min_child_weight=1,

            subsample=0.8,
            colsample_bytree=value,

            n_estimators=3000,

            tree_method="hist",

            random_state=42,

            early_stopping_rounds=300,
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False,
        )

        pred = model.predict_proba(X_valid)[:, 1]

        oof_pred[valid_idx] = pred

    score = log_loss(y, oof_pred)

    results.append({
        "colsample_bytree": value,
        "logloss": score,
    })

    print(f"OOF LogLoss : {score:.6f}")
    print()

results = (
    pd.DataFrame(results)
    .sort_values("logloss")
    .reset_index(drop=True)
)

display(results)

colsample_bytree = 0.5
OOF LogLoss : 0.544232

colsample_bytree = 0.6
OOF LogLoss : 0.543698

colsample_bytree = 0.7
OOF LogLoss : 0.543818

colsample_bytree = 0.8
OOF LogLoss : 0.543822

colsample_bytree = 0.9
OOF LogLoss : 0.544365

colsample_bytree = 1.0
OOF LogLoss : 0.544623



,colsample_bytree,logloss
0,0.6,0.543698
1,0.7,0.543818
2,0.8,0.543822
3,0.5,0.544232
4,0.9,0.544365
5,1.0,0.544623


**learning_rate = 0.02  
max_depth = 7  
min_child_weight = 1  
subsample = 0.8  
colsample_bytree = 0.6**